In [0]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.append(os.path.abspath("../src"))
from pyspark.sql import functions as F
from legacy_ceded import ceded_claims

In [0]:
dbutils.widgets.text("catalog", "reinsurance")
catalog = dbutils.widgets.get("catalog")

claims = spark.table(f"{catalog}.staging.stg_claims")
treaties = spark.table(f"{catalog}.staging.stg_treaties")
as_of = claims.agg(F.max("load_date")).first()[0]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.legacy")
ceded_claims(claims, treaties, as_of).write.mode("overwrite").saveAsTable(f"{catalog}.legacy.ceded_claims")
print(as_of, spark.table(f"{catalog}.legacy.ceded_claims").count())

In [0]:
display(spark.sql(f"""
WITH vault AS (
  SELECT h.claim_number, t.treaty_number, c.loss_date, c.gross_incurred, c.ceded_incurred
  FROM {catalog}.business_vault.claim_ceded c
  JOIN {catalog}.raw_vault.hub_claim h ON h.hk_claim = c.hk_claim
  JOIN {catalog}.raw_vault.hub_treaty t ON t.hk_treaty = c.hk_treaty
  WHERE c.snapshot_date = (SELECT max(snapshot_date) FROM {catalog}.business_vault.claim_ceded)
),
legacy AS (
  SELECT claim_number, count(*) AS rows, sum(gross_incurred) AS gross_incurred, sum(ceded_incurred) AS ceded_incurred
  FROM {catalog}.legacy.ceded_claims
  GROUP BY claim_number
)
SELECT CASE
         WHEN l.claim_number IS NULL THEN 'dropped by legacy: no treaty'
         WHEN v.claim_number IS NULL THEN 'only in legacy'
         WHEN v.gross_incurred = l.gross_incurred AND v.ceded_incurred = l.ceded_incurred THEN 'match'
         WHEN l.rows > 1 THEN 'duplicate rows in bordereau'
         WHEN v.treaty_number = 'T-CED01-MOTOR-2024' AND v.loss_date < '2024-07-01' THEN 'endorsement applied retroactively'
         ELSE 'unexplained'
       END AS reason,
       count(*) AS claims,
       sum(coalesce(l.gross_incurred, 0) - coalesce(v.gross_incurred, 0)) AS gross_diff,
       sum(coalesce(l.ceded_incurred, 0) - coalesce(v.ceded_incurred, 0)) AS ceded_diff
FROM vault v
FULL JOIN legacy l ON l.claim_number = v.claim_number
GROUP BY 1
ORDER BY 1"""))